In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **IMPORT LIBRARY**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans

# **AQUIRE DATA**

In [ ]:
file_path = "/content/drive/MyDrive/Data/"

In [ ]:
reader = pd.read_csv(file_path + 'Data.csv', chunksize=1000000)

In [ ]:
bet = pd.concat([chunk for chunk in reader])

## **DEFINE FUNCTION**

In [ ]:
def create_data_by_year_filter(dataframe, year):
  return dataframe[(dataframe['DATE_DIM'] >= f'{year}-01-01') & (dataframe['DATE_DIM'] <= f'{year}-12-31')]

In [ ]:
def calculate_recency(dataframe, year):
  ref_date = f'{year}-12-31'
  dataframe['DATE_DIM'] = pd.to_datetime(dataframe['DATE_DIM'])
  ref_date = pd.to_datetime(ref_date)
  recency = pd.DataFrame(dataframe.groupby(['BET_ACCOUNT_NUM_HASH']).agg({'DATE_DIM' : lambda x : ((ref_date - x.max()).days)}).reset_index(drop=True))
  recency.rename(columns={'DATE_DIM': 'Recency'}, inplace=True)
  return recency

In [ ]:
def calculate_frequency(dataframe):
  freq = pd.DataFrame(dataframe.groupby('BET_ACCOUNT_NUM_HASH')['DATE_DIM'].count()).reset_index(drop=True)
  freq.rename(columns={'DATE_DIM': 'Frequency'}, inplace=True)
  return freq

In [ ]:
def calculate_monetary(dataframe):
  monetary = pd.DataFrame(dataframe.groupby('BET_ACCOUNT_NUM_HASH')['TOTAL_TURNOVER'].sum()).reset_index(drop=True)
  monetary.rename(columns={'TOTAL_TURNOVER': 'Monetary'}, inplace=True)
  return monetary

In [ ]:
def calculate_RFM(dataframe, year):
  df_year = create_data_by_year_filter(dataframe, year)
  recency = calculate_recency(df_year, year)
  frequency = calculate_frequency(df_year)
  monetary = calculate_monetary(df_year)
  return recency, frequency, monetary, df_year

In [ ]:
def capping_outlier(value, maximum, minimum):
  if value > maximum:
    return maximum
  elif value < minimum:
    return minimum
  else:
    return value

In [ ]:
def calculate_Q1_median_Q3(dataframe, column):
  Q1 = np.percentile(dataframe[column], 25)
  Q3 = np.percentile(dataframe[column], 75)
  median = np.median(dataframe[column])
  return Q1, median, Q3

In [ ]:
def calculate_upper_limit_lower_limit(dataframe, column, Q1, Q3):
  maximum = float(Q3 + 1.5 * (Q3 - Q1))
  minimum = float(np.min(dataframe[column]))
  return maximum, minimum

In [ ]:
def calculate_score(dataframe, column, Q1, median, Q3):
  if column == 'Recency':
    dataframe[f'{column}_Score'] = dataframe[column].apply(lambda x: 4 if x <= Q1 else (3 if x <= median else (2 if x <= Q3 else 1)))
  else:
    dataframe[f'{column}_Score'] = dataframe[column].apply(lambda x: 1 if x <= Q1 else (2 if x <= median else (3 if x <= Q3 else 4)))
  return dataframe

In [ ]:
def process_columns(dataframe):
  for column in dataframe.columns:
    Q1, median, Q3 = calculate_Q1_median_Q3(dataframe, column)
    maximum, minimum = calculate_upper_limit_lower_limit(dataframe, column, Q1, Q3)
    dataframe[column] = dataframe[column].apply(capping_outlier, args=(maximum, minimum))
    dataframe = calculate_score(dataframe, column, Q1, median, Q3)
  return dataframe

In [ ]:
def create_RFM(dataframe, year):
  recency, frequency, monetary, df_year = calculate_RFM(dataframe, year)
  RFM = pd.concat([recency, frequency, monetary], axis = 1)
  RFM = process_columns(RFM)
  RFM['Overall_Score'] = RFM[['Recency_Score', 'Frequency_Score', 'Monetary_Score']].sum(axis=1)
  Overall_score = pd.DataFrame(RFM.groupby('Overall_Score')['Recency', 'Frequency', 'Monetary'].mean())
  RFM["BET_ACCOUNT_NUM_HASH"] = list(df_year.groupby('BET_ACCOUNT_NUM_HASH').groups.keys())
  if year == '2021':
    RFM['Segment'] = RFM['Overall_Score'].apply(segment_score_2021)
  elif year == '2022':
    RFM['Segment'] = RFM['Overall_Score'].apply(segment_score_2022)
  RFM.drop(columns = ['Recency', 'Frequency',	'Monetary', 'Recency_Score', 'Frequency_Score' ,'Monetary_Score', 'Overall_Score']	, inplace = True)
  return RFM, Overall_score

In [ ]:
def segment_score_2021(value):
  if value > 10:
    return 'Champions'
  elif value > 8:
    return 'Loyal Customers'
  elif value > 5:
    return 'About to sleep'
  elif value > 3:
    return 'Hibernating'
  else:
    return 'Lost'

def segment_score_2022(value):
  if value > 10:
    return 'Champions'
  elif value > 8:
    return 'Loyal Customers'
  elif value > 3:
    return 'About to sleep'
  else:
    return 'Hibernating'

#### **RFM MODEL 2021**




In [ ]:
RFM_2021, overall_score_2021 = create_RFM(bet, '2021')

<ipython-input-9-b5a9f839e8db>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe['DATE_DIM'] = pd.to_datetime(dataframe['DATE_DIM'])
<ipython-input-40-03e98c3b82b3>:6: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  Overall_score = pd.DataFrame(RFM.groupby('Overall_Score')['Recency', 'Frequency', 'Monetary'].mean())


#### **RFM MODEL 2022**

In [ ]:
RFM_2022, overall_score_2022 = create_RFM(bet, '2022')

<ipython-input-9-b5a9f839e8db>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe['DATE_DIM'] = pd.to_datetime(dataframe['DATE_DIM'])
<ipython-input-40-03e98c3b82b3>:6: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  Overall_score = pd.DataFrame(RFM.groupby('Overall_Score')['Recency', 'Frequency', 'Monetary'].mean())


## **MERGE TABLE**

In [ ]:
merge_2021= pd.merge(RFM_2021, bet[(bet['DATE_DIM'] >= '2021-01-01') & (bet['DATE_DIM'] <= '2021-12-31')], on='BET_ACCOUNT_NUM_HASH', how = 'right')
merge_2022 = pd.merge(RFM_2022, bet[(bet['DATE_DIM'] >= '2022-01-01') & (bet['DATE_DIM'] <= '2022-12-31')], on='BET_ACCOUNT_NUM_HASH', how = 'right')

In [ ]:
merge = pd.merge(merge_2021, merge_2022, how = 'outer')

In [ ]:
file_path = "/content/drive/MyDrive/RFM Model/"

In [ ]:
reader = pd.read_csv(file_path + 'RFM data.csv', chunksize=1000000)

In [ ]:
merge = pd.concat([chunk for chunk in reader])

In [ ]:
merge

,Segment,BET_ACCOUNT_NUM_HASH,DATE_DIM,DAY_OF_WEEK,AGE,AGE_BAND,GENDER,TENURE_IN_DAYS,RESIDENTIAL_STATE,FOB_RACING_TURNOVER,FOB_SPORT_TURNOVER,PARI_RACING_TURNOVER,PARI_SPORT_TURNOVER,TOTAL_TURNOVER,DIVIDENDS_PAID,GROSS_MARGIN,TICKETS
0,Champions,13154,2021-01-01,Fri,67.0,65+,M,11846,WA,37.0,NaN,1081.0,NaN,1118.0,443.55,271.254275,288
1,Champions,18379,2021-01-01,Fri,54.0,45-54,M,1884,WA,40.0,NaN,NaN,NaN,40.0,0.00,40.000000,1
2,Champions,559232,2021-01-01,Fri,63.0,55-64,M,2866,WA,NaN,NaN,12.0,NaN,12.0,9.50,2.041720,5
3,Champions,698904,2021-01-01,Fri,69.0,65+,M,2100,WA,NaN,NaN,1223.5,NaN,1223.5,267.91,245.117147,40
4,Champions,762921,2021-01-01,Fri,67.0,65+,M,4766,WA,NaN,NaN,17.5,NaN,17.5,0.00,3.504075,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12364096,Champions,4293715592,2022-12-31,Sat,61.0,55-64,F,6871,WA,28.0,NaN,53.0,NaN,81.0,103.10,38.770481,33
12364097,Loyal Customers,4294296954,2022-12-31,Sat,30.0,25-34,U,14,OTH,450.0,NaN,NaN,NaN,450.0,475.00,-25.000000,9
12364098,Loyal Customers,4294378139,2022-12-31,Sat,46.0,45-54,U,2062,WA,5.0,NaN,NaN,NaN,5.0,0.00,5.000000,1
12364099,Loyal Customers,4294561160,2022-12-31,Sat,33.0,25-34,F,3560,WA,70.0,NaN,18.0,NaN,88.0,0.00,73.659467,12


In [ ]:
bet_dashboard = merge[['DATE_DIM', 'DAY_OF_WEEK', 'BET_ACCOUNT_NUM_HASH', 'AGE_BAND', 'GENDER', 'RESIDENTIAL_STATE', 'FOB_RACING_TURNOVER', 'FOB_SPORT_TURNOVER', 'PARI_RACING_TURNOVER', 'PARI_SPORT_TURNOVER', 'TOTAL_TURNOVER', 'GROSS_MARGIN', 'Segment']]

In [ ]:
bet_dashboard.to_csv(file_path + 'OVERVIEW_DETAIL.csv', index=False)

## **EXPORT DATA**

In [ ]:
merge.to_csv('/content/drive/MyDrive/RFM Model/RFM data.csv', index = False)